# Applied ML, Industrial — XGBoost with SHAP and a Monitoring Sketch

**Category:** Applied ML, Industrial · XGBoost, ECNN Pump Monitoring.

**What this notebook does.** Fits an XGBoost classifier on a synthetic loan-default dataset, evaluates with the metrics that actually matter in production (calibration, top-decile lift), computes SHAP-style feature importances, and ends with a sketch of what *production monitoring* should track on top of it.

**The point.** The training cell is the smallest part of a real applied-ML system. Most of the value is in the loop around it.

## 1. Setup — synthetic tabular loan dataset

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(7)
N = 5000
income      = rng.lognormal(10.5, 0.6, N)
debt_ratio  = rng.beta(2, 5, N) * 2
tenure_yrs  = rng.exponential(5, N)
missed_pmts = rng.poisson(0.5, N)
region      = rng.integers(0, 4, N)
# Hidden ground-truth — default risk
logit = (
    -3.0
    + 1.4 * debt_ratio
    + 0.6 * missed_pmts
    - 0.6 * np.log(income / 30000 + 1)
    - 0.05 * tenure_yrs
    + 0.15 * (region == 2)
)
p = 1 / (1 + np.exp(-logit))
y = (rng.random(N) < p).astype(int)
X = pd.DataFrame({'income': income, 'debt_ratio': debt_ratio,
                  'tenure_yrs': tenure_yrs, 'missed_pmts': missed_pmts,
                  'region': region})
print('default rate:', y.mean().round(3))


## 2. Train with proper CV

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier  # XGBoost stand-in (no extra dep)
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)
clf = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=7)
clf.fit(X_tr, y_tr)
p_te = clf.predict_proba(X_te)[:, 1]
print(f'AUC:   {roc_auc_score(y_te, p_te):.3f}')
print(f'LogL:  {log_loss(y_te, p_te):.3f}')
print(f'Brier: {brier_score_loss(y_te, p_te):.3f}')


## 3. Calibration — the metric most ML teams forget

AUC tells you ordering. Calibration tells you whether the *probability* is meaningful. Pricing, capital reserves, and reserves regulations all depend on calibration, not AUC.

In [ ]:
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(y_te, p_te, n_bins=10)
plt.figure(figsize=(4.5,4))
plt.plot([0,1],[0,1],'k--', alpha=0.4)
plt.plot(mean_pred, frac_pos, 'o-')
plt.xlabel('predicted prob'); plt.ylabel('observed frac')
plt.title('Calibration curve')
plt.tight_layout(); plt.show()


## 4. Top-decile lift — what risk teams actually look at

In [ ]:
ord_idx = np.argsort(-p_te)
top10 = ord_idx[: len(p_te)//10]
lift = (y_te.iloc[top10].mean() if hasattr(y_te,'iloc') else y_te[top10].mean()) / y_te.mean()
print(f'Top-decile lift: {lift:.2f}× the base rate')


## 5. Feature importance — permutation, not just gain

In [ ]:
from sklearn.inspection import permutation_importance
imp = permutation_importance(clf, X_te, y_te, n_repeats=5, random_state=7)
ranking = sorted(zip(X.columns, imp.importances_mean), key=lambda x: -x[1])
for name, val in ranking:
    print(f'  {name:14s} {val:.4f}')


## 6. Production monitoring sketch — what to alert on

In [ ]:
# Simulate three months of production: distribution drift in debt_ratio
drift_X = X_te.copy()
drift_X['debt_ratio'] = drift_X['debt_ratio'] + 0.15  # macro economic shift
p_drifted = clf.predict_proba(drift_X)[:, 1]

import pandas as pd
rows = []
for name, baseline, current in [
    ('mean predicted prob', p_te.mean(), p_drifted.mean()),
    ('debt_ratio mean',     X_te['debt_ratio'].mean(),  drift_X['debt_ratio'].mean()),
    ('debt_ratio p95',      X_te['debt_ratio'].quantile(0.95), drift_X['debt_ratio'].quantile(0.95)),
]:
    rows.append({'metric': name, 'baseline': round(baseline,3), 'current': round(current,3),
                 'delta': round(current - baseline, 3)})
pd.DataFrame(rows)


## My take

The training cell is six lines. The rest of this notebook is the actual job:

1. **Calibration is a board-level metric** in any regulated industry. A perfectly-ranking model with bad calibration mis-prices risk by definition.
2. **Top-decile lift** is the metric a non-technical stakeholder can reason about. Frame model wins in lift, not AUC, when you talk to the business.
3. **Permutation importance beats gain-based importance** for the question 'which features can I lose access to.' Gain tells you what the tree *used*; permutation tells you what *matters at inference.*
4. **Drift monitoring is your insurance policy.** Most production model failures aren't sudden — they're the slow drift of an input distribution while AUC on a stale evaluation set stays flat.

When I interview an applied ML candidate, I ask them to walk through the *monitoring plan* for a model they've shipped. The answer separates the ML engineers who own production from the ones who throw notebooks over a fence.